In [5]:
# numpy and pandas
import numpy as np
import pandas as pd

# train/test split
from sklearn.model_selection import train_test_split

# preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Random Forest
from sklearn.ensemble import RandomForestClassifier

# Cross-validation
from sklearn.model_selection import StratifiedKFold, cross_val_score

# Hyperparameter tuning
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

# Save model
import joblib

# Time
import time

# Reproducibility
np.random.seed(42)

print("Ready to go!")


# -----------------------------
# CREATE DATA
# -----------------------------

n_students = 300

hours_studied = np.clip(
    np.random.normal(5, 2.3, n_students), 0, 10
).round(1)

attendance_pct = np.clip(
    np.random.normal(75, 18, n_students), 0, 100
).round(0)

previous_score = np.clip(
    np.random.normal(65, 15, n_students), 0, 100
).round(0)

study_method = np.random.choice(
    ["Group", "Solo", "Online"],
    size=n_students,
    p=[0.4, 0.4, 0.2]
)

combined_score = (
    0.5 * (hours_studied / 10)
    + 0.3 * (attendance_pct / 100)
    + 0.2 * (previous_score / 100)
)

noisy_score = combined_score + np.random.normal(
    0, 0.13, n_students
)

passed = (noisy_score > 0.5).astype(int)


# -----------------------------
# CREATE DATAFRAME
# -----------------------------

students = pd.DataFrame({
    "hours_studied": hours_studied,
    "attendance_pct": attendance_pct,
    "previous_score": previous_score,
    "study_method": study_method,
    "passed": passed
})


# -----------------------------
# ADD MISSING VALUES
# -----------------------------

for column, missing_fraction in [
    ("hours_studied", 0.08),
    ("attendance_pct", 0.05),
    ("previous_score", 0.08),
    ("study_method", 0.10)
]:

    rows_to_blank = students.sample(
        frac=missing_fraction,
        random_state=1
    ).index

    students.loc[rows_to_blank, column] = np.nan


print("Missing values per column:")
print(students.isna().sum())

print("\nFirst 10 rows:")
print(students.head(10))


# -----------------------------
# TRAIN / TEST SPLIT
# -----------------------------

X = students.drop(columns=["passed"])
y = students["passed"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("\nTraining rows:", len(X_train))
print("Test rows:", len(X_test))


# -----------------------------
# PREPROCESSING
# -----------------------------

numeric_features = [
    "hours_studied",
    "attendance_pct",
    "previous_score"
]

categorical_features = [
    "study_method"
]


numeric_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler())
])


categorical_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore"))
])


preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

print("\nPreprocessor built!")


# -----------------------------
# RANDOM FOREST PIPELINE
# -----------------------------

model_pipeline = Pipeline([
    ("prep", preprocessor),
    ("clf", RandomForestClassifier(random_state=42))
])


# -----------------------------
# BASELINE MODEL
# -----------------------------

model_pipeline.fit(X_train, y_train)

baseline_accuracy = model_pipeline.score(
    X_test,
    y_test
)

print(
    f"Baseline test accuracy: {baseline_accuracy:.1%}"
)


# -----------------------------
# CROSS VALIDATION
# -----------------------------

cv_splitter = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

fold_scores = cross_val_score(
    model_pipeline,
    X_train,
    y_train,
    cv=cv_splitter
)

print(
    "Score on each of the 5 folds:",
    fold_scores.round(3)
)

print(
    f"Mean accuracy: {fold_scores.mean():.1%}"
)

print(
    f"Standard deviation: {fold_scores.std():.1%}"
)


# -----------------------------
# GRID SEARCH
# -----------------------------

param_grid = {
    "clf__n_estimators": [100, 200],
    "clf__max_depth": [4, 8, None],
    "clf__min_samples_split": [2, 5]
}

start_time = time.time()

grid_search = GridSearchCV(
    model_pipeline,
    param_grid,
    cv=cv_splitter,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

grid_search_seconds = time.time() - start_time

combinations = (
    len(param_grid["clf__n_estimators"])
    * len(param_grid["clf__max_depth"])
    * len(param_grid["clf__min_samples_split"])
)

print(
    f"Grid Search checked {combinations} combinations"
)

print(
    f"Best cross-validated accuracy: "
    f"{grid_search.best_score_:.1%}"
)

print(
    "Best settings found:",
    grid_search.best_params_
)

print(
    f"Time taken: {grid_search_seconds:.2f} seconds"
)


# -----------------------------
# RANDOM SEARCH
# -----------------------------

param_distribution = {
    "clf__n_estimators": [50, 100, 150, 200, 300],
    "clf__max_depth": [3, 4, 5, 6, 8, 10, None],
    "clf__min_samples_split": [2, 3, 5, 8, 10],
    "clf__min_samples_leaf": [1, 2, 4]
}

start_time = time.time()

random_search = RandomizedSearchCV(
    model_pipeline,
    param_distribution,
    n_iter=10,
    cv=cv_splitter,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

random_search_seconds = time.time() - start_time

print(
    "Random Search checked only 10 combinations "
    "out of 525 possible"
)

print(
    f"Best cross-validated accuracy: "
    f"{random_search.best_score_:.1%}"
)

print(
    "Best settings found:",
    random_search.best_params_
)

print(
    f"Time taken: {random_search_seconds:.2f} seconds"
)


# -----------------------------
# FINAL MODEL
# -----------------------------

final_model = grid_search.best_estimator_

final_test_accuracy = final_model.score(
    X_test,
    y_test
)

print(
    f"\nBaseline (untuned) test accuracy: "
    f"{baseline_accuracy:.1%}"
)

print(
    f"Final (tuned) test accuracy:      "
    f"{final_test_accuracy:.1%}"
)


# -----------------------------
# SAVE MODEL
# -----------------------------

joblib.dump(
    final_model,
    "student_pass_predictor.joblib"
)

print(
    "\nModel saved to student_pass_predictor.joblib"
)

Ready to go!
Missing values per column:
hours_studied     24
attendance_pct    15
previous_score    24
study_method      30
passed             0
dtype: int64

First 10 rows:
   hours_studied  attendance_pct  previous_score study_method  passed
0            6.1            60.0            76.0        Group       1
1            4.7            65.0            51.0       Online       1
2            6.5            88.0            78.0       Online       1
3            8.5            86.0            85.0         Solo       1
4            4.5            75.0            71.0        Group       1
5            4.5            77.0            93.0        Group       1
6            8.6            98.0            53.0         Solo       1
7            6.8            64.0            46.0       Online       1
8            3.9            85.0            38.0         Solo       1
9            6.2            71.0            87.0        Group       1

Training rows: 225
Test rows: 75

Preprocessor built!
B

In [4]:
%%writefile app.py

import streamlit as st
import pandas as pd
import joblib


# Load the trained model
model = joblib.load("student_pass_predictor.joblib")


# Page title
st.title("Will This Student Pass?")


# Description
st.write(
    "Enter a student's details below, "
    "and the model will predict pass or fail."
)


# Student inputs
hours_studied = st.slider(
    "Hours studied",
    0.0,
    10.0,
    5.0
)

attendance_pct = st.slider(
    "Attendance percent",
    0.0,
    100.0,
    75.0
)

previous_score = st.slider(
    "Previous test score",
    0.0,
    100.0,
    65.0
)

study_method = st.selectbox(
    "Study method",
    ["Group", "Solo", "Online"]
)


# Prediction button
if st.button("Predict"):

    # Create input DataFrame
    input_row = pd.DataFrame([{
        "hours_studied": hours_studied,
        "attendance_pct": attendance_pct,
        "previous_score": previous_score,
        "study_method": study_method
    }])


    # Make prediction
    prediction = model.predict(input_row)[0]


    # Get probability
    probability = model.predict_proba(input_row)[0][1]


    # Display result
    if prediction == 1:

        st.success(
            f"Prediction: PASS "
            f"(confidence: {probability:.0%})"
        )

    else:

        st.error(
            f"Prediction: FAIL "
            f"(confidence: {1 - probability:.0%})"
        )

Writing app.py


In [6]:
import sklearn
print(sklearn.__version__)

1.6.1
